Bu notebookda feature extraction yapılacaktır. answer_making'de üretilen her cevap için token seviyesinde zaman serilerinden sabit boyutlu feature vektörleri çıkarılacak.
ver_500_ma.json da 4 ham metrik 4 tane de mean ortalam metrikleri vardır.

Denemelerde gördüğüm üzere ilk aşamada çok fazla feature çıkardım ancak daha sonrasında bazı featureların belli bir eşiğin altında etki ettiğini gördüm.
Bundan dolayı feature eliminasyonu yaptım.
Bazı featureları buna benzer bir çalışmadan direk alıp kullandım. Onlayı daha sonrasında ayrıntılı olarka feature eliminasyon işleminde bahsedeceğim.
Kendi belirlediğim featureları da ek olarak açıklayacağım.

In [ ]:
import json
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# answer_making.ipynb'de üretilen veri yüklenmekte
with open('/content/drive/MyDrive/proje5/data/veri_500_ma.json', encoding='utf-8') as f:
    veri = json.load(f)

egitim = veri['egitim']
test   = veri['test']

print(f"Eğitim soru : {len(egitim)}")
print(f"Test soru   : {len(test)}")

# Bir cevabın hangi metrikleri içerdiğini kontrol et
ornek = egitim[0]['cevaplar'][0]
print(f"\nBir cevabın anahtarları:")
for k, v in ornek.items():
    if isinstance(v, list):
        print(f"  {k:20s} → liste, {len(v)} eleman")
    else:
        print(f"  {k:20s} → {v}")

Feature extraction işleminde 4 adet metrik bulunmakta bunlar şu şekildedir.
1. ma_top10_sums -> burada top 10 değerinin ortalaması 
2. ma_top1_probs -> sadece en yüksek olasılık ortalaması
3. ma_top1_top2_fark -> en yüksek 2 olasılık arasındaki fark ortalaması
4. ma_entropy -> modelin emin olup olmadığını belirlemek için kullanılan bir matematiksel yaklaşım "H=−i∑​pi​⋅log(pi​)"

Bu metrikleri kullanarak featurelar hazırlanmıştır, aşağıda featurelar ve açıklamaları yer almaktadır.

1. mean -> ortalama
2. std -> standart sapma
3. min -> minimum 
4. max -> maksimum
5. ilk ve son %33 kısımlarındaki ortalamalar.
6. ilk token - son token ilişkisi
7. eşik oran altında kalan tokenlar
8. makaleden alınan featurelar -> violation count, monotonicity, max delta, final entropy
9. Türev featureları -> birinci, ikinci türev ve dönüm noktaları
10. token uzunlukları 
11. komşu tokenlar arasındaki fark ortalaması (dalgalanmayı ölçmek için)
12. A* algoritmasında yer alan heuristic benzer bir yapıda ani düşüş şiddeti ve pozisyonu ölçme
13. kritik yükselişin olduğu an nerede olduğunu belirleme


In [ ]:
import numpy as np
from scipy.stats import spearmanr

# Bu method ile XGBoost için sabit boyutlu bir feature vektörü çıkarma işlemi yapıldı.
def feature_cıkar(cevap):
    """
    Bir cevaptan XGBoost için sabit boyutlu feature vektörü çıkarır.
    Değişken uzunluktaki token serilerini istatistiksel özetlerle temsil eder.
    """
    t10 = np.array(cevap['ma_top10_sums'])
    t1  = np.array(cevap['ma_top1_probs'])
    t12 = np.array(cevap['ma_top1_top2_fark'])
    ent = np.array(cevap['ma_entropy'])
    n   = len(t10)

    ilk = max(1, int(n * 0.33))
    son = max(1, int(n * 0.67))

    features = {}

    # Bu döngüden 4 x 22 = 88 adet feature elde ediliyor. 
    for isim, seri in [('t10', t10), ('t1', t1), ('t12', t12), ('ent', ent)]:

        # ── 1. İstatistiksel özetler ──────────────────────
        features[f'{isim}_mean'] = float(np.mean(seri))
        features[f'{isim}_std']  = float(np.std(seri))
        features[f'{isim}_min']  = float(np.min(seri))
        features[f'{isim}_max']  = float(np.max(seri))

        # ── 2. Bölümsel ortalamalar ───────────────────────
        features[f'{isim}_ilk_mean'] = float(np.mean(seri[:ilk]))
        features[f'{isim}_son_mean'] = float(np.mean(seri[son:]))

        # ── 3. Trend ─────────────────────────────────────
        # Pozitif → ilerledikçe artıyor
        # Negatif → ilerledikçe azalıyor (yanlış cevap sinyali) Ai ile belirledim.
        features[f'{isim}_trend'] = float(np.mean(seri[son:]) - np.mean(seri[:ilk]))

        # ── 4. Dalgalanma (MADD) ──────────────────────────
        # Komşu tokenlar arası ortalama mutlak fark Ai ile belirledim.
        # Yüksek → model sürekli sıçrıyor, tutarsız
        farklar = np.abs(np.diff(seri))
        features[f'{isim}_madd'] = float(np.mean(farklar))

        # ── 5. A* heuristic ───────────────────────────────
        # A* algoritmasındaki "hedefe uzaklık" mantığına benzer şekilde
        # modelin doğru cevaptan saptığı anları yakalamak için
        # burada top10, top1 ve top1-top2 farkının yüksek sapmaların olması modelin emin olması anlamında geliyor.
        if isim in ['t10', 't1', 't12']:
            # Bu metrikler için yüksek = iyi → düşüş = sapma
            sapmalar = np.maximum(0, -np.diff(seri))
        # burada ise entropy değerinde azaliş bu sefer modelin daha emin olduğu anlamına geliyor. 
        else:
            # Entropy için düşük = iyi → artış = sapma
            sapmalar = np.maximum(0, np.diff(seri))

        features[f'{isim}_max_drop']        = float(np.max(sapmalar))
        features[f'{isim}_drop_pos']        = float(np.argmax(sapmalar) / n)
        features[f'{isim}_cumulative_drop'] = float(np.sum(sapmalar))

        #  9. Türev feature'ları 
        d1 = np.diff(seri)           # birinci türev — değişim hızı
        d2 = np.diff(d1)             # ikinci türev  — ivme

        # Birinci türev istatistikleri
        features[f'{isim}_d1_mean'] = float(np.mean(d1))   # ortalama hız
        features[f'{isim}_d1_std']  = float(np.std(d1))    # hız dalgalanması
        features[f'{isim}_d1_max']  = float(np.max(d1))    # en hızlı artış
        features[f'{isim}_d1_min']  = float(np.min(d1))    # en hızlı düşüş

        # İkinci türev istatistikleri
        features[f'{isim}_d2_mean'] = float(np.mean(d2))   # ortalama ivme
        features[f'{isim}_d2_std']  = float(np.std(d2))    # ivme dalgalanması
        features[f'{isim}_d2_max']  = float(np.max(d2))    # en büyük ivmelenme
        features[f'{isim}_d2_min']  = float(np.min(d2))    # en büyük yavaşlama

        # Dönüm noktaları — ikinci türevin işaret değiştirdiği yerler
        # Modelin "yön değiştirdiği" anları temsil eder, + -> 1 ile - ise -> -1 ile temsil ediliyor.
        isaretler      = np.sign(d2)
        # Dönüm olduğunda 0 dan farklı bir değer dönüyor 2 veya -2 gibi
        donum          = np.where(np.diff(isaretler) != 0)[0]
        features[f'{isim}_donum_sayisi']   = float(len(donum) / n)
        # Dönğm olmayan dizilerde boş dizi olacağından hata veriyor ondan dolayı 0.5 nötr değer dönüyor.
        features[f'{isim}_ilk_donum_pos']  = float(donum[0] / n  if len(donum) > 0 else 0.5)
        features[f'{isim}_son_donum_pos']  = float(donum[-1] / n if len(donum) > 0 else 0.5)

    # Döngü dışından 11 adet feature elde ediliyor.
    # ── 6. Eşik oranları ──────────────────────────────────
    features['ent_yuksek_oran'] = float(np.sum(ent > 1.0) / n)
    features['ent_dusuk_oran']  = float(np.sum(ent < 0.2) / n)
    features['t10_dusuk_oran']  = float(np.sum(t10 < 0.985) / n)
    features['t12_dusuk_oran']  = float(np.sum(t12 < 0.5) / n)

    # ── 7. Pozisyonel ─────────────────────────────────────
    features['ent_max_pos'] = float(np.argmax(ent) / n)
    features['t12_min_pos'] = float(np.argmin(t12) / n)

    # ── 8. Makale feature'ları ────────────────────────────
    # Kaynak: "Entropy trajectory shape predicts LLM reasoning reliability"
    ent_diff = np.diff(ent)

    # Entropy'nin azaldığı adım oranı (monotonluk ihlali)
    features['ent_violation_count']    = float(np.sum(ent_diff < 0) / n)

    # Tek adımda en büyük entropy artışı
    features['ent_max_positive_delta'] = float(np.max(np.maximum(0, ent_diff)))

    # Entropy ile pozisyon arasındaki Spearman korelasyonu
    # +1 = mükemmel monoton artış, -1 = monoton azalış
    rho, _ = spearmanr(np.arange(n), ent)
    features['ent_monotonicity']       = float(rho)

    # Son token entropy değeri 
    features['ent_final'] = float(ent[-1])

    # ── 10. Uzunluk ───────────────────────────────────────
    # Pilot analizde yanlış cevaplar daha uzundu (253 vs 219 token)
    features['num_tokens'] = float(cevap['num_tokens'])

    return features

# Toplam 99 adet featuremuz var artık 99 boyutlu bir vektörümüz var buna göre eğitim yapıulacak.
# Test: Tek cevaptan feature çıkar
test_f = feature_cıkar(egitim[0]['cevaplar'][0])
print(f"Feature sayısı : {len(test_f)}")
print(f"Doğru mu       : {egitim[0]['cevaplar'][0]['dogru_mu']}")

In [ ]:
# Tüm cevaplara feature_cıkar uygulanarak X ve y matrisleri oluşturulmakta.
# X: her satır bir cevabın feature vektörü
# y: 1=doğru cevap, 0=yanlış cevap
feature_isimleri = list(feature_cıkar(egitim[0]['cevaplar'][0]).keys())

X_train, y_train = [], []
X_test,  y_test  = [], []

# Eğitim seti
# Sırasıyla sorular alınıyor, ardından bir alt döngüde o sorunun 10 cevabı alınıyor. 
# Daha sonrasında her sorunun cevabına bu çıkarılan featurelar dictiornayler oluşturuluryor.
# çıktıya da bu cevabın doğru olup olmadığı bilgisi yazılıyor.
for soru in egitim:
    for cevap in soru['cevaplar']:
        X_train.append(list(feature_cıkar(cevap).values())) # Feature vektörleri 
        y_train.append(1 if cevap['dogru_mu'] else 0)       # etiketler 

# Test seti
for soru in test:
    for cevap in soru['cevaplar']:
        X_test.append(list(feature_cıkar(cevap).values()))
        y_test.append(1 if cevap['dogru_mu'] else 0)

# numpy arrayine dönüşüm, python listinden array çevirmemizin nedeni yukarıda yer alan append kısmı daha hızlı oluyor python listesinde
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test  = np.array(X_test)
y_test  = np.array(y_test)

# test ve trainde var olan doğru ve yanlışların görmek için
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape} — Doğru: {y_train.sum()}, Yanlış: {(y_train==0).sum()}")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape} — Doğru: {y_test.sum()}, Yanlış: {(y_test==0).sum()}")

In [ ]:
# Drive kayıt işlemleri kodunu ai ile yaptım.
# Matrisler ve feature isimleri Drive'a kaydedilmekte.
# Bu dosyalar model_training.ipynb tarafından kullanılacaktır.
np.save('/content/drive/MyDrive/proje5/data/X_train.npy', X_train)
np.save('/content/drive/MyDrive/proje5/data/y_train.npy', y_train)
np.save('/content/drive/MyDrive/proje5/data/X_test.npy',  X_test)
np.save('/content/drive/MyDrive/proje5/data/y_test.npy',  y_test)

with open('/content/drive/MyDrive/proje5/data/feature_isimleri.json', 'w') as f:
    json.dump(feature_isimleri, f, indent=2)

print(f"✅ Kaydedildi:")
print(f"  X_train.npy, y_train.npy")
print(f"  X_test.npy,  y_test.npy")
print(f"  feature_isimleri.json ({len(feature_isimleri)} feature)")